# DifficultyJudge — Standalone Training, Optimization & Eval

Self-contained notebook for the stripped-down DifficultyJudge module.

**Outputs only:** `predicted_cefr` + `predicted_difficulty`  
**Input format:** every question must have a `question_id` field (e.g. `"Q1"`)  
**Datasets:** `gepa_train.jsonl` (training) · `gold_good.jsonl` + `gold_flawed.jsonl` (Promptfoo eval only)  
**Flow:** Setup → Define Agent → Load Data → Baseline Eval → GEPA Optimize → Post-Eval → Promptfoo

Run all cells top-to-bottom. No changes are made to any existing `.py` files.

## Cell 1 — Setup & Imports

In [ ]:
import sys, os, json, subprocess, warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')

# ── Locate project root (directory that contains utils.py) ──────────────────
_candidate = Path().resolve()
if (_candidate / 'utils.py').exists():
    PROJECT_ROOT = _candidate
elif (_candidate.parent / 'utils.py').exists():
    PROJECT_ROOT = _candidate.parent
else:
    raise RuntimeError(
        f'Cannot locate project root from {_candidate}. '
        'Open Jupyter from d:/Topin or d:/Topin/notebooks.'
    )

NOTEBOOK_DIR  = PROJECT_ROOT / 'notebooks'
DATA_DIR      = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
EVALS_DIR     = NOTEBOOK_DIR / 'evals'

ARTIFACTS_DIR.mkdir(exist_ok=True)
EVALS_DIR.mkdir(parents=True, exist_ok=True)

OPTIMIZED_PATH = ARTIFACTS_DIR / 'simple_difficulty_optimized.json'
EVAL_OUTPUT    = EVALS_DIR / 'difficulty_eval_output.json'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Auto-inject venv site-packages so dspy is importable from any kernel ────
_venv_sp = PROJECT_ROOT / '.venv' / 'Lib' / 'site-packages'      # Windows
if not _venv_sp.exists():
    _venv_sp = PROJECT_ROOT / '.venv' / 'lib' / 'site-packages'  # Linux/macOS
if _venv_sp.exists() and str(_venv_sp) not in sys.path:
    sys.path.insert(0, str(_venv_sp))
    print(f'Injected venv site-packages: {_venv_sp}')
else:
    print(f'Venv site-packages path: {_venv_sp}  (exists={_venv_sp.exists()})')

import dspy

print(f'Project root : {PROJECT_ROOT}')
print(f'Artifacts    : {ARTIFACTS_DIR}')
print(f'Evals dir    : {EVALS_DIR}')
print(f'DSPy version : {dspy.__version__}')

## Cell 2 — Configure DSPy (Mistral)

In [ ]:
from utils import configure_dspy_from_env

lm = configure_dspy_from_env()
print(f'Active LM : {lm}')

## Cell 3 — Define Models + Signature + Agent

- `Question` — typed input model (one MCQ question, all fields)
- `DifficultyResult` — typed output model (`question_id`, `predicted_cefr`, `predicted_difficulty`)
- `SimpleDifficultySignature` — `questions: list[Question]` → `results: list[DifficultyResult]`

In [ ]:
from pydantic import BaseModel
from typing import Literal


# ── Input model ──────────────────────────────────────────────────────────────
class Question(BaseModel):
    question_id:       str
    stem:              str
    options:           list[str]
    correct_answer:    str
    explanation:       str
    target_cefr:       str
    target_difficulty: str


# ── Output model ─────────────────────────────────────────────────────────────
class DifficultyResult(BaseModel):
    question_id:          str
    predicted_cefr:       Literal['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
    predicted_difficulty: Literal['Easy', 'Medium', 'Hard']


class DifficultyOutput(BaseModel):
    results: list[DifficultyResult]


# ── Signature ────────────────────────────────────────────────────────────────
class SimpleDifficultySignature(dspy.Signature):
    """Classify a list of MCQ questions by CEFR level and difficulty.

    For EACH question in the input list, analyse four dimensions:
      1. vocabulary_level      — A1=basic (go, eat), B1=everyday-formal (routine, consistent),
                                 C1=academic (discrepancy, implication)
      2. grammar_complexity    — A1=simple present, B1=mixed tenses/conditionals,
                                 C1=inversion/embedded clauses
      3. reasoning_load        — Low=recall/recognition (A1-A2), Medium=inference (B1-B2),
                                 High=multi-step logic (C1-C2)
      4. distractor_difficulty — Low=clearly wrong options, High=semantically subtle options

    Map the combination of all four dimensions to a CEFR band:
      A1/A2 → Easy   |   B1/B2 → Medium   |   C1/C2 → Hard

    Return one DifficultyResult per question, in the same order as the input list.
    """
    questions: list[Question] = dspy.InputField(desc='List of MCQ questions to be classified')
    output:    DifficultyOutput = dspy.OutputField(desc='Classification results for all questions')


# ── Agent ─────────────────────────────────────────────────────────────────────
class SimpleDifficultyAgent(dspy.Module):
    """Batch difficulty judge. Attribute `judge` serialises as `judge.predict` in the artifact."""

    def __init__(self):
        super().__init__()
        self.judge = dspy.ChainOfThought(SimpleDifficultySignature)

    def forward(self, questions: list[Question]) -> dspy.Prediction:
        return self.judge(questions=questions)


print('Models and agent defined.')
print('  Input  : questions: list[Question]')
print('  Output : output: DifficultyOutput')
print('           output.results → list[DifficultyResult]')
print('           each result: question_id | predicted_cefr | predicted_difficulty')

## Cell 4 — Define Metric

Score is **0.0**, **0.5**, or **1.0** per question based on CEFR + difficulty match.  
Returns `(float, str)` tuple for GEPA — plain `float` for BootstrapFewShot.

In [ ]:
# CEFR → DifficultyBand mapping
DIFF_MAP = {
    'A1': 'Easy',   'A2': 'Easy',
    'B1': 'Medium', 'B2': 'Medium',
    'C1': 'Hard',   'C2': 'Hard',
}


def difficulty_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    """
    Reads pred.output (DifficultyOutput) → pred.output.results[0] (DifficultyResult).
    GEPA: 5 args → returns (float, str).
    BootstrapFewShot: 3 args → returns plain float.
    """
    try:
        result = pred.output.results[0]
        predicted_cefr = result.predicted_cefr
        predicted_diff = result.predicted_difficulty
    except Exception:
        msg = 'No valid output from prediction.'
        return (0.0, msg) if pred_name is not None else 0.0

    expected_cefr = gold.expected_predicted_cefr
    expected_diff = DIFF_MAP.get(expected_cefr, gold.target_difficulty)

    cefr_match = int(predicted_cefr == expected_cefr)
    diff_match = int(predicted_diff == expected_diff)
    score = (cefr_match + diff_match) / 2.0

    feedback = (
        f'Expected predicted_cefr={expected_cefr} (got "{predicted_cefr}"), '
        f'expected predicted_difficulty={expected_diff} (got "{predicted_diff}"). '
        f'Score: {score:.1f}. '
        + ('Both correct.' if score == 1.0
           else 'Match CEFR to vocabulary complexity, grammar, and reasoning load.')
    )

    return (score, feedback) if pred_name is not None else score


print('difficulty_metric defined  (reads pred.output.results[0]).')

## Cell 5 — Load Datasets

| File | Role | Questions |
|------|------|-----------|
| `data/training_dataset_standard.json` | Optimizer training | 24 (4 per CEFR A1–C2) |
| `data/eval_dataset_standard.json` | Promptfoo eval only | 24 (4 per CEFR A1–C2) |

**Required fields per record:** `question_id`, `stem`, `options`, `correct_answer`, `explanation`, `target_cefr`, `target_difficulty`

In [ ]:
def load_dataset(path: Path) -> list:
    """Load a JSON or JSONL file into dspy.Example objects.
    `questions` field holds a list[Question] — typed input for the agent.
    If `target_difficulty` is null/missing it is derived from `target_cefr` via DIFF_MAP.
    """
    path = Path(path)
    if path.suffix == '.json':
        rows = json.loads(path.read_text(encoding='utf-8'))
    else:
        rows = []
        with path.open('r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))

    examples = []
    for i, row in enumerate(rows, 1):
        question_id = row.get('question_id', f'Q{i}')
        target_cefr = row['target_cefr']
        # Fall back to DIFF_MAP if target_difficulty is null or missing
        target_difficulty = row.get('target_difficulty') or DIFF_MAP.get(target_cefr, 'Medium')

        q = Question(
            question_id=question_id,
            stem=row['stem'],
            options=row['options'],
            correct_answer=row['correct_answer'],
            explanation=row['explanation'],
            target_cefr=target_cefr,
            target_difficulty=target_difficulty,
        )

        examples.append(
            dspy.Example(
                question_id=question_id,
                questions=[q],
                target_cefr=target_cefr,
                target_difficulty=target_difficulty,
                expected_predicted_cefr=row.get('expected_predicted_cefr', target_cefr),
                expected_overall_decision=row.get('expected_overall_decision', 'Pass'),
            ).with_inputs('questions')
        )
    return examples


# Training dataset (optimizer only)
trainset = load_dataset(DATA_DIR / 'training_dataset_standard.json')

# Eval dataset (Promptfoo only — never used for training)
evalset  = load_dataset(DATA_DIR / 'eval_dataset_standard.json')

print(f'Training dataset : {len(trainset)} examples')
print(f'Eval dataset     : {len(evalset)} examples')
print()

cefr_dist    = Counter(ex.target_cefr for ex in trainset)
cefr_dist_ev = Counter(ex.target_cefr for ex in evalset)
print(f'Training CEFR distribution: {dict(sorted(cefr_dist.items()))}')
print(f'Eval     CEFR distribution: {dict(sorted(cefr_dist_ev.items()))}')
print()

ex = trainset[0]
q0 = ex.questions[0]
print(f'Sample — question_id     : {q0.question_id}')
print(f'         target_cefr     : {q0.target_cefr}')
print(f'         target_difficulty: {q0.target_difficulty}')
print(f'         stem            : {q0.stem[:80]}...')
print(f'         options         : {q0.options}')

## Cell 6 — Baseline Evaluation (Before Optimization)

In [ ]:
def evaluate_agent(agent, dataset, label=''):
    """Run agent on every example. Reads pred.output.results[0] (DifficultyResult)."""
    records = []
    for ex in dataset:
        try:
            pred   = agent(questions=ex.questions)
            score  = difficulty_metric(ex, pred)
            result = pred.output.results[0] if pred.output and pred.output.results else None
        except Exception as e:
            pred, score, result = None, 0.0, None
            print(f'  [ERROR] {ex.question_id}: {e}')

        records.append({
            'question_id': ex.question_id,
            'gold':        ex,
            'result':      result,   # DifficultyResult or None
            'score':       score,
        })

    avg = sum(r['score'] for r in records) / len(records) if records else 0.0
    print(f'[{label}]  N={len(records)}  avg_score={avg:.3f}')
    return records, avg


baseline_agent = SimpleDifficultyAgent()
print('Running baseline evaluation on training dataset...')
baseline_records, baseline_score = evaluate_agent(baseline_agent, trainset, label='Baseline')

print()
print(f'{"Q-ID":<6} {"target":<8} {"expected":<10} {"pred_cefr":<10} {"pred_diff":<12} {"score":>5}')
print('-' * 58)
for r in baseline_records:
    gold = r['gold']
    res  = r['result']
    print(f"{r['question_id']:<6} {gold.target_cefr:<8} {gold.expected_predicted_cefr:<10} "
          f"{(res.predicted_cefr if res else 'ERR'):<10} "
          f"{(res.predicted_difficulty if res else 'ERR'):<12} "
          f"{r['score']:>5.1f}")
print('-' * 58)
print(f'Baseline avg : {baseline_score:.3f}')

## Cell 7 — Run Optimizer

Tries **GEPA** first — rewrites the prompt instruction using a reflection LLM so the judge gets better at predicting CEFR.  
Falls back to **BootstrapFewShot** if GEPA raises any error.  
Saves the optimized agent to `artifacts/simple_difficulty_optimized.json`.

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path=PROJECT_ROOT / '.env')


def run_gepa(trainset):
    from dspy.teleprompt import GEPA

    reflection_lm = dspy.LM(
        f"openai/{os.getenv('MISTRAL_MODEL', 'open-mistral-nemo')}",
        api_key=os.environ['MISTRAL_API_KEY'],
        api_base=os.getenv('MISTRAL_API_BASE', 'https://api.mistral.ai/v1'),
        temperature=0.9,
        max_tokens=4000,
    )

    log_dir = ARTIFACTS_DIR / 'gepa_difficulty_logs'
    log_dir.mkdir(exist_ok=True)

    optimizer = GEPA(
        metric=difficulty_metric,
        auto='light',                    # ~6 candidate prompts
        reflection_lm=reflection_lm,
        reflection_minibatch_size=3,
        log_dir=str(log_dir),
        track_stats=True,
        seed=42,
    )

    student = SimpleDifficultyAgent()
    split = max(2, int(len(trainset) * 0.8))
    print(f'  GEPA: {split} train / {len(trainset) - split} val examples')

    optimized = optimizer.compile(
        student,
        trainset=trainset[:split],
        valset=trainset[split:],
    )
    return optimized, 'GEPA'


def run_bootstrap(trainset):
    from dspy.teleprompt import BootstrapFewShot
    optimizer = BootstrapFewShot(
        metric=difficulty_metric,
        max_bootstrapped_demos=3,
        max_labeled_demos=4,
    )
    student = SimpleDifficultyAgent()
    print(f'  BootstrapFewShot: {len(trainset)} examples')
    optimized = optimizer.compile(student, trainset=trainset)
    return optimized, 'BootstrapFewShot'


# ── Try GEPA, fall back to BootstrapFewShot ────────────────────────────────
print('Attempting GEPA optimization...')
try:
    optimized_agent, optimizer_used = run_gepa(trainset)
    print('GEPA complete.')
except Exception as gepa_err:
    print(f'GEPA failed: {gepa_err}')
    print('Falling back to BootstrapFewShot...')
    optimized_agent, optimizer_used = run_bootstrap(trainset)
    print('BootstrapFewShot complete.')

optimized_agent.save(str(OPTIMIZED_PATH))
print(f'\nOptimizer used : {optimizer_used}')
print(f'Artifact saved : {OPTIMIZED_PATH}')

## Cell 8 — Load Optimized Agent & Post-Optimization Evaluation

In [ ]:
loaded_agent = SimpleDifficultyAgent()
loaded_agent.load(str(OPTIMIZED_PATH))
print(f'Loaded optimized agent from {OPTIMIZED_PATH}')

print('\nRunning post-optimization evaluation on training dataset...')
optimized_records, optimized_score = evaluate_agent(loaded_agent, trainset, label='Optimized')

print()
print(f'{"Q-ID":<6} {"target":<8} {"expected":<10} {"pred_cefr":<10} {"pred_diff":<12} {"score":>5}')
print('-' * 58)
for r in optimized_records:
    gold = r['gold']
    res  = r['result']
    print(f"{r['question_id']:<6} {gold.target_cefr:<8} {gold.expected_predicted_cefr:<10} "
          f"{(res.predicted_cefr if res else 'ERR'):<10} "
          f"{(res.predicted_difficulty if res else 'ERR'):<12} "
          f"{r['score']:>5.1f}")
print('-' * 58)
print(f'Baseline avg  : {baseline_score:.3f}')
print(f'Optimized avg : {optimized_score:.3f}')
print(f'Delta         : {optimized_score - baseline_score:+.3f}')

## Cell 10 — Write Promptfoo Provider

Standalone Python file imported by Promptfoo.  
Accepts a JSON array of question objects, runs the optimized judge, returns the predictions array.

In [ ]:
provider_code = '''\
from __future__ import annotations
import json, os, sys
from pathlib import Path
from typing import Literal
from pydantic import BaseModel

EVALS_DIR    = Path(__file__).resolve().parent
PROJECT_ROOT = EVALS_DIR.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

_venv_sp = PROJECT_ROOT / \'.venv\' / \'Lib\' / \'site-packages\'
if not _venv_sp.exists():
    _venv_sp = PROJECT_ROOT / \'.venv\' / \'lib\' / \'site-packages\'
if _venv_sp.exists() and str(_venv_sp) not in sys.path:
    sys.path.insert(0, str(_venv_sp))

from utils import configure_dspy_from_env
import dspy


class Question(BaseModel):
    question_id:       str
    stem:              str
    options:           list[str]
    correct_answer:    str
    explanation:       str
    target_cefr:       str
    target_difficulty: str


class DifficultyResult(BaseModel):
    question_id:          str
    predicted_cefr:       Literal[\'A1\', \'A2\', \'B1\', \'B2\', \'C1\', \'C2\']
    predicted_difficulty: Literal[\'Easy\', \'Medium\', \'Hard\']


class DifficultyOutput(BaseModel):
    results: list[DifficultyResult]


class SimpleDifficultySignature(dspy.Signature):
    """Classify a list of MCQ questions by CEFR level and difficulty.
    Map: A1/A2 -> Easy | B1/B2 -> Medium | C1/C2 -> Hard.
    Return one DifficultyResult per question in the same order.
    """
    questions: list[Question]   = dspy.InputField(desc=\'List of MCQ questions to be classified\')
    output:    DifficultyOutput = dspy.OutputField(desc=\'Classification results for all questions\')


class SimpleDifficultyAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.judge = dspy.ChainOfThought(SimpleDifficultySignature)

    def forward(self, questions: list[Question]) -> dspy.Prediction:
        return self.judge(questions=questions)


_OPTIMIZED = PROJECT_ROOT / "artifacts" / "simple_difficulty_optimized.json"
_agent = None


def call_api(prompt: str, options, context):
    global _agent
    configure_dspy_from_env()

    if _agent is None:
        _agent = SimpleDifficultyAgent()
        if _OPTIMIZED.exists():
            _agent.load(str(_OPTIMIZED))

    try:
        data = json.loads(prompt)
        if isinstance(data, dict):
            data = [data]
        questions = [Question(**q) for q in data]
    except Exception as e:
        return {"error": f"Invalid input: {e}"}

    try:
        pred    = _agent(questions=questions)
        results = [r.model_dump() for r in pred.output.results]
        return {"output": json.dumps(results)}
    except Exception as e:
        return {"error": str(e)}
'''

provider_path = EVALS_DIR / 'difficulty_eval_provider.py'
provider_path.write_text(provider_code, encoding='utf-8')
print(f'Written: {provider_path}')

## Cell 11 — Build Promptfoo Test Cases

Source: `eval_dataset_standard.json` (24 questions, 4 per CEFR level A1–C2).  
Each test case passes one question as a JSON array; asserts `predicted_cefr` and `predicted_difficulty` match `target_cefr`.

In [ ]:
def build_tests(examples: list) -> list:
    """Build Promptfoo test cases from dspy.Example objects.
    question_json = JSON array of one Question dict (as expected by the provider).
    """
    tests = []
    for ex in examples:
        expected_cefr = ex.expected_predicted_cefr
        expected_diff = DIFF_MAP.get(expected_cefr, ex.target_difficulty)

        # Serialise the Question object to a JSON array of dicts
        q = ex.questions[0]
        question_json = json.dumps([q.model_dump()])

        tests.append({
            'description': f'{ex.question_id} | target={ex.target_cefr} expect={expected_cefr}',
            'vars': {'question_json': question_json},
            'assert': [
                {
                    'type': 'javascript',
                    'value': (
                        f'const results = JSON.parse(output); '
                        f'const p = Array.isArray(results) ? results[0] : results; '
                        f'return p.predicted_cefr === "{expected_cefr}";'
                    ),
                },
                {
                    'type': 'javascript',
                    'value': (
                        f'const results = JSON.parse(output); '
                        f'const p = Array.isArray(results) ? results[0] : results; '
                        f'return p.predicted_difficulty === "{expected_diff}";'
                    ),
                },
            ],
        })
    return tests


all_tests = build_tests(evalset)

print(f'Total eval test cases : {len(all_tests)}  (from eval_dataset_standard.json)')
cefr_dist_tests = Counter(ex.target_cefr for ex in evalset)
print(f'CEFR distribution     : {dict(sorted(cefr_dist_tests.items()))}')
print()
for t in all_tests:
    print(f'  {t["description"]}')

## Cell 12 — Write Promptfoo Config YAML

In [ ]:
import yaml

config = {
    'description': 'DifficultyJudge eval — predicted_cefr + predicted_difficulty (with question_id)',
    'providers': [{'id': 'file://difficulty_eval_provider.py'}],
    'prompts': ['{{question_json}}'],
    'tests': all_tests,
}

config_path = EVALS_DIR / 'difficulty_eval_config.yaml'
with config_path.open('w', encoding='utf-8') as f:
    yaml.dump(config, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print(f'Written: {config_path}')

## Cell 13 — Run Promptfoo Eval as Subprocess

In [ ]:
cmd = [
    'npx', 'promptfoo@latest', 'eval',
    '-c', str(config_path),
    '-o', str(EVAL_OUTPUT),
    '--output-format', 'json',
    '--no-cache',
    '--max-concurrency', '1',
]

print('Running Promptfoo eval...')
print(f'  Config : {config_path}')
print(f'  Output : {EVAL_OUTPUT}')
print()

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    shell=True,            # required on Windows: npx is a .cmd file
    cwd=str(EVALS_DIR),
    timeout=600,
)

print('--- STDOUT (last 3000 chars) ---')
stdout = result.stdout
print(stdout[-3000:] if len(stdout) > 3000 else stdout)

if result.returncode != 0:
    print('--- STDERR ---')
    stderr = result.stderr
    print(stderr[-2000:] if len(stderr) > 2000 else stderr)
    print(f'\nPromptfoo exited with code {result.returncode}')
else:
    print('\nPromptfoo completed successfully (exit code 0).')

## Cell 14 — Parse & Display Promptfoo Results

In [ ]:
if not EVAL_OUTPUT.exists():
    print(f'Output file not found: {EVAL_OUTPUT}')
    print('Re-run the Promptfoo cell above.')
else:
    raw_text = EVAL_OUTPUT.read_text(encoding='utf-8').strip()
    if not raw_text:
        print(f'Output file is empty: {EVAL_OUTPUT}')
        print('Promptfoo likely errored before writing results.')
        print('Check the STDOUT / STDERR printed in the cell above.')
    else:
        try:
            eval_data = json.loads(raw_text)
        except json.JSONDecodeError as e:
            print(f'Could not parse output file: {e}')
            print('First 500 chars of file:')
            print(raw_text[:500])
        else:
            raw          = eval_data.get('results', {})
            results_list = raw.get('results', raw) if isinstance(raw, dict) else raw
            stats        = raw.get('stats', {}) if isinstance(raw, dict) else {}

            passed = sum(1 for r in results_list if r.get('success', False))
            total  = len(results_list)

            print(f'Promptfoo Eval Results')
            print(f'  Passed : {passed}/{total}')
            print(f'  Failed : {total - passed}/{total}')
            if stats:
                print(f'  Stats  : {stats}')
            print()

            print(f'{"Q-ID":<10} {"Description":<45} {"Result":<7} {"pred_cefr":<11} {"pred_diff"}')
            print('-' * 88)

            for r in results_list:
                desc    = r.get('description', '')[:44]
                success = 'PASS' if r.get('success') else 'FAIL'
                raw_out = r.get('response', {}).get('output', '[]')
                try:
                    p_list = json.loads(raw_out)
                    p = p_list[0] if isinstance(p_list, list) and p_list else p_list
                    qid       = p.get('question_id', 'N/A')
                    pred_cefr = p.get('predicted_cefr', 'N/A')
                    pred_diff = p.get('predicted_difficulty', 'N/A')
                except Exception:
                    qid = pred_cefr = pred_diff = 'PARSE_ERR'

                print(f'{qid:<10} {desc:<45} {success:<7} {pred_cefr:<11} {pred_diff}')

            print()
            print(f'Full eval output saved to: {EVAL_OUTPUT}')

---
## Summary of All Outputs

| File | Created when | Contents |
|------|-------------|----------|
| `artifacts/simple_difficulty_optimized.json` | Cell 7 | Optimized judge with improved prompt / few-shot demos |
| `artifacts/gepa_difficulty_logs/` | Cell 7 (GEPA) | Per-candidate prompt scores and logs |
| `notebooks/evals/difficulty_eval_provider.py` | Cell 10 | Promptfoo provider (auto-written) |
| `notebooks/evals/difficulty_eval_config.yaml` | Cell 12 | Promptfoo test config (auto-written) |
| `notebooks/evals/difficulty_eval_output.json` | Cell 13 | Full Promptfoo results per question |

---
## Input Format Reference

Every record in every dataset must follow this format:

```json
{
  "question_id":             "Q1",
  "stem":                    "She _______ to school every day.",
  "options":                 ["go", "goes", "going", "gone"],
  "correct_answer":          "goes",
  "explanation":             "Third-person singular subjects require -s/-es.",
  "target_cefr":             "A1",
  "target_difficulty":       "Easy",
  "expected_predicted_cefr": "A1"
}
```

**Training only** (`gepa_train.jsonl`) — also include:
```json
  "expected_overall_decision": "Pass"
```

---
## Re-use the Trained Judge in Future Sessions

```python
agent = SimpleDifficultyAgent()
agent.load('artifacts/simple_difficulty_optimized.json')

pred   = agent(
    stem="She _______ to school every day.",
    options=json.dumps(["go", "goes", "going", "gone"]),
    correct_answer="goes",
    explanation="Third-person singular subjects require -s/-es.",
    target_cefr="A1",
    target_difficulty="Easy",
)
result = json.loads(pred.output_json)
# result = {"predicted_cefr": "A1", "predicted_difficulty": "Easy"}
```